# Session 1: What a machine means when it says it learns

**259.075 Data-integrated Algorithmic Design Processes II** · Session 1 of 7

<img src="img/wien-zinshaus.jpg" width="560">

*An apartment house from around 1880, Lange Gasse 25, Vienna. Photo: Funke,
[Wikimedia Commons](https://commons.wikimedia.org/wiki/File:1080_Lange_Gasse_25.jpg),
CC BY-SA 4.0.*

How much heating does a building need? Working it out properly takes an afternoon of simulation.
Today you build a **model**, a simple rule that predicts one number from another, and let it learn
the heating demand of 1073 Vienna buildings. The rule inside it will be found by a few lines that
**you** wrote. One honest warning up front: our heating demand is itself computed, not measured;
the box below says why.

No maths beyond adding, subtracting, multiplying and dividing. Every new idea comes with a small
example first.

<div style="border-left: 5px solid #d8a24a; background-color: #fdf6e3; color: #222; padding: 10px 16px; margin: 6px 0;">

**💡 About the data**

The buildings and their construction periods come from the open data of the City of Vienna. The
other columns are ours: area, perimeter and compactness are measured from each outline, the
U-value of the wall is a textbook value per construction period, and the heating demand is
**computed** from those two, because no open data gives measured consumption per building. Keep
that in mind; it matters at the end.

<details>
<summary>The column names are German. Click for a translation.</summary>

| column | German | meaning |
|---|---|---|
| `id` | | the city's number for the building |
| `bauperiode` | Bauperiode | construction period; *vor* means before, *nach* after |
| `startjahr` | Startjahr | a year taken from the period's name, used for colouring |
| `bautyp` | Bautyp | the city's code for the type of building |
| `flaeche_m2` | Fläche | footprint area in m² |
| `umfang_m` | Umfang | perimeter of the footprint in m |
| `kompaktheit` | Kompaktheit | compactness, see step 2 |
| `u_wert` | U-Wert | U-value of the wall in W/m²K: how much heat passes through it; higher means leakier |
| `hwb` | Heizwärmebedarf | heating demand in kWh per m² and year, the computed column |
| `lat`, `lon` | | position on the map |

</details>
</div>

## Step 1: load the buildings

Click into the cell below and press **Shift + Enter**. That runs it. Do the same with every code
cell, from top to bottom.

In [ ]:
import numpy as np                    # numbers and lists of numbers
import pandas as pd                   # tables
import plotly.express as px           # quick interactive plots
import plotly.graph_objects as go     # plots built piece by piece

df = pd.read_csv("../data/gebaeude_wien.csv")   # load the table; df stands for "data frame"

print(f"✓ {len(df)} buildings loaded. Here are the first five rows:")   # a friendly check
df.head()                             # show the first five rows of the table

## Step 2: look at them

The next plot shows one dot per building: **compactness** across, **heating demand** up.

Compactness is the perimeter divided by the square root of the area. The square root of 100 is
10, because 10 × 10 = 100. So a square house of 10 × 10 m, with a perimeter of 40 m and an area of
100 m², gets 40 m / 10 m = 4. The metres cancel, so compactness has no unit. A circle gets about 3.5, the lowest possible; a long, thin slab gets much
more. **A higher number means a less compact shape.** The square root makes the number the same
for a small and a large house of the same shape, so a shed and a block can be compared.

The **U-value** (`u_wert`) says how much heat passes through a wall: the higher, the leakier. A
wall with U = 1.50 lets twice as much heat through as one with U = 0.75. The heating demand is
given in kWh/m²a: kilowatt-hours per square metre of floor and year (*a* for *annum*).

Run the cell and hover over a few dots.

In [ ]:
# scatter plot: compactness across, heating demand up, coloured by construction period
fig = px.scatter(df, x="kompaktheit", y="hwb", color="startjahr",     # one dot per building
                 hover_data=["id", "bauperiode", "flaeche_m2", "u_wert"],
                 labels={"kompaktheit": "compactness (perimeter / sqrt area)",
                         "hwb": "heating demand, computed [kWh/m2a]",
                         "startjahr": "period from"},
                 color_continuous_scale="Viridis", height=520)         # colours and plot height
fig.update_traces(marker=dict(size=7, opacity=0.75))   # dot size and transparency
fig.show()                                             # draw it

Find two dots with almost the same compactness but clearly different heights, and compare their
`u_wert` in the hover box. Why is their heating demand different?

<details>
<summary>Answer</summary>

Their U-values differ. Two houses of the same shape with different walls need different amounts
of heating. That is why the dots sit in several bands, one per U-value, instead of on one line.

</details>

## Step 3: find a building you know

The same buildings on a map. It covers four parts of Vienna. Look for one you know, where you
live, the institute, a corner you pass every day, hover over it and copy its `id` into the cell
after the map. Give it five minutes; if none of yours is there, pick any that catches your eye.

In [ ]:
# map: one dot per building, colour = heating demand, size = footprint area
fig = px.scatter_map(df, lat="lat", lon="lon", color="hwb", size="flaeche_m2",   # dots on a map
                     color_continuous_scale="Turbo", zoom=11, height=520,
                     hover_data=["id", "bauperiode", "kompaktheit", "u_wert"],
                     map_style="carto-positron")     # a light street map underneath
fig.show()                                            # draw it

In [ ]:
# YOUR CHOICE: paste the id of your building here, instead of None
meine_id = None                                       # meine_id: "my id"

if meine_id is None:                                  # nothing pasted yet
    print("No building chosen yet. Paste an id from the map above, then run this cell again.")  # a reminder
elif meine_id not in set(df["id"]):                   # a number that is not in the table
    print(f"There is no building with id {meine_id} in the table. Check the number on the map.")  # a hint
else:                                                 # found it
    mein_haus = df[df["id"] == meine_id].iloc[0]      # mein_haus: "my house", its row in the table
    print(f"✓ Your building {meine_id}: built {mein_haus['bauperiode']}, "    # its details, as a sentence
          f"{mein_haus['flaeche_m2']:.0f} m² footprint, compactness {mein_haus['kompaktheit']:.2f}, "
          f"heating demand {mein_haus['hwb']:.0f} kWh/m²a. We come back to it at the end.")

## Step 4: a line as a model

Our model is a straight line through the cloud:

    predicted demand = w × compactness + b

- `w` is the line's **slope**: how much the predicted demand rises when compactness goes up by one.
- `b` lifts or lowers the whole line.

An example: with w = 8 and b = 110, a house with compactness 5 gets 8 × 5 + 110 = 150 kWh/m²a.
Raise `b` to 112, and the same house gets 152: the whole line moved up by 2.
A 100 m² flat at 150 kWh/m²a would need 15 000 kWh of heating a year.

In the language of the field, this line is a **neuron**: an input (compactness), a **weight**
(`w`), a **bias** (`b`), an output. Every network later in this course is built from neurons
like this one.

In [ ]:
x = df["kompaktheit"].to_numpy()      # x: the compactness of all 1073 buildings
y = df["hwb"].to_numpy()              # y: their heating demand

w, b = 8, 110                         # one example line
print(f"✓ For compactness 5 the line predicts {w * 5 + b} kWh/m²a.")   # 8 × 5 + 110
print(f"  For the first building in the table (compactness {x[0]:.2f}) it predicts "  # x[0]: first building
      f"{w * x[0] + b:.0f}; the table says {y[0]:.0f}.")

## Step 5: one number for "how wrong"

To find a good line, we need a way to say how wrong a line is. Take three made-up houses and a
flat line that predicts 155 for all of them:

| invented house | demand in the table | line says | miss = line − table | miss squared |
|---|---|---|---|---|
| A | 150 | 155 | +5 | 25 |
| B | 160 | 155 | −5 | 25 |
| C | 140 | 155 | +15 | 225 |

We square every miss, so that misses above and below do not cancel out, and so that one big miss
counts more than several small ones. Then we average: (25 + 25 + 225) / 3 = 91.7.

> This average of the squared misses is the **mean squared error**, short **MSE**. The smaller,
> the better the line.

In [ ]:
def mse(w, b, x, y):                  # a recipe with four ingredients: the line (w, b) and the data (x, y)
    """Mean squared error of the line  predicted = w * x + b."""
    vorhersage = w * x + b            # what the line predicts for every house (Vorhersage: prediction)
    fehler = vorhersage - y           # the miss for every house (Fehler: error)
    return float(np.mean(fehler ** 2))   # square every miss, then take the average


drei_haeuser = np.array([150.0, 160.0, 140.0])       # drei Häuser: the three invented houses' demand
egal = np.zeros(3)                                   # egal: "doesn't matter"; x has no effect when w = 0
print(f"✓ MSE for the three houses: {mse(0, 155, egal, drei_haeuser):.1f}, "   # w = 0, b = 155
      f"as computed by hand above")

## Step 6: your line, by hand

Run the cell below. Under the plot there is a **slider marked "w ="**. Drag it and watch the red
line move; `b` stays at 110. The title shows the MSE of the line for all 1073 buildings. Find the
smallest MSE you can and write it down.

In [ ]:
# slider plot: one frame per whole-number w from 0 to 30, b fixed at 110
xs = np.linspace(x.min(), x.max(), 50)                # 50 points along the compactness axis
b_fix = 110.0                                         # b stays fixed
ws = np.arange(0, 31, 1)                              # the slider positions: 0, 1, ..., 30

frames = [go.Frame(name=f"{w}",                         # one picture per slider position
                   data=[go.Scatter(x=xs, y=w * xs + b_fix, mode="lines",
                                    line=dict(width=4, color="crimson"))],
                   traces=[1],
                   layout=go.Layout(title=f"w = {w},  b = {b_fix:.0f},  MSE = {mse(w, b_fix, x, y):.0f}"))
          for w in ws]

fig = go.Figure(                                       # the plot: buildings as dots, the line in red
    data=[go.Scatter(x=x, y=y, mode="markers", name="buildings",
                     marker=dict(size=6, opacity=0.5, color="#4c78a8")),
          go.Scatter(x=xs, y=ws[0] * xs + b_fix, mode="lines", name="your line",
                     line=dict(width=4, color="crimson"))],
    frames=frames)
fig.update_layout(                                     # axis titles, title, and the slider itself
    height=560, xaxis_title="compactness", yaxis_title="heating demand, computed",
    title=f"w = {ws[0]},  b = {b_fix:.0f},  MSE = {mse(ws[0], b_fix, x, y):.0f}",
    sliders=[dict(active=0, currentvalue=dict(prefix="w = "),
                  steps=[dict(method="animate", label=f"{w}",
                              args=[[f"{w}"], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
                         for w in ws])])
fig.show()                                             # draw it

<details>
<summary>Answer</summary>

At w = 8 the title shows an MSE of 53. One step left or right, and it gets worse. That is the best
whole-number setting with b held at 110; the very best `w` lies a little above 8.

</details>

<img src="img/linear-regression.png" width="380">

*Finding the best line through a cloud of points is over two hundred years old: Legendre and
Gauss published it around 1805 as the method of least squares. Figure: Sewaqu,
[Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Linear_regression.svg), public
domain.*

## Step 7: which way is better?

You found the best `w` by trying. A computer needs a rule. Here are the MSE values your slider
showed around 5:

| w | 4 | 5 | 6 |
|---|---|---|---|
| MSE | 652 | 405 | 222 |

From 5, going right lowers the MSE. So the rule says: make `w` bigger. Near 9 it is the other way
round.

The computer measures this with a tiny step instead of a whole one. At w = 5 the MSE is
404.527550; at w = 5.001 it is 404.313032. It changed by −0.214518 for a step of 0.001, so per
whole unit of `w` that is −0.214518 / 0.001 ≈ −214.5. This number, how strongly the MSE changes per unit you move, is
called the **gradient**. Big and negative means "going right helps a lot".

> To improve, step **against** the gradient: if it is negative, increase `w`; if positive,
> decrease it. Repeating that is called **gradient descent**.

Try it yourself in the next cell: type any `w`, run, and the cell tells you which way to go.
Play with it until it says "you are at the bottom". After three or four tries, feel free to open
the answer below.

In [ ]:
# YOUR CHOICE: any value for w, for example 5, 12 or 8.3
mein_w = 5                                            # mein_w: "my w"

klein = 0.001                                         # a tiny step (klein: small)
vorher = mse(mein_w, 110, x, y)                       # MSE where you stand (vorher: before)
nachher = mse(mein_w + klein, 110, x, y)              # MSE one tiny step to the right (nachher: after)
gradient_w = (nachher - vorher) / klein               # change in MSE per unit of w

print(f"MSE at w = {mein_w}: {vorher:.1f}, gradient {gradient_w:.1f}")   # where you stand
if gradient_w < -1:                                   # clearly going down to the right
    print("→ make w bigger")                          # advice: step right
elif gradient_w > 1:                                  # clearly going up to the right
    print("← make w smaller")                         # advice: step left
else:                                                 # almost flat
    print("✓ you are at the bottom: the gradient is almost zero")   # done

<details>
<summary>Answer</summary>

The bottom is at about w = 8.3, where the MSE is 50.5, a little better than the 53 of the best
whole-number setting. Notice that the gradient shrinks as you get closer: about −215 at 5, about
−20 at 8. Near the bottom the ground is flat.

</details>

## Step 8: your part, the learning loop

Now the computer does it by itself, thousands of small steps, adjusting `w` **and** `b`. One step:

    new w = old w − learning rate × gradient of w

The **learning rate** sets the step size. An example with small numbers: w = 10, gradient 4,
learning rate 0.1. The new w is 10 − 0.1 × 4 = 9.6.

The first cell below gives the loop a helper, `gradienten`, that works out two gradients at once:
one for `w` and one for `b`, the second found the same way, holding `w` still and nudging `b`. It
uses a shortcut formula you do not need to read; the cell prints its results next to the tiny-step
method from step 7. For `w` they differ in the last digit shown, because a tiny step is a very
good but not perfect approximation.

The second cell is the loop. Two lines in it are yours: the comments say exactly what to write.
The German names: `lernrate` is the learning rate, `schritte` the number of steps, `verlauf` the
history of all steps.

The same rule moves `b`, with its own gradient: b = 110, gradient −36, learning rate 0.1 gives
110 − 0.1 × (−36) = 113.6, so the line moves up.

How to write that in Python: the step example above, w = 10 − 0.1 × 4, becomes
`w = 10 - 0.1 * 4`. The `=` means "from now on, w is", `-` is minus and `*` is times. In the loop
you write the same thing with the names instead of the numbers.

Stuck? Ask your neighbour or the tutor right away, or open the answer below the cell.

In [ ]:
# Helper for the loop. You do not need to read the formula; it gives almost the same answer as tiny steps.
def gradienten(w, b, x, y):                           # a recipe: line (w, b) and data (x, y) in, two gradients out
    """Both gradients of the MSE at (w, b), with a shortcut formula instead of tiny steps."""
    fehler = w * x + b - y                            # the miss of every house
    gradient_w = 2 * np.mean(fehler * x)              # how the MSE changes per unit of w
    gradient_b = 2 * np.mean(fehler)                  # how the MSE changes per unit of b
    return float(gradient_w), float(gradient_b)       # hand both numbers back


kurz_w, kurz_b = gradienten(5, 110, x, y)                               # kurz: the shortcut, at w = 5, b = 110
probe_w = (mse(5 + 0.001, 110, x, y) - mse(5, 110, x, y)) / 0.001       # probe: tiny step in w, b held still
probe_b = (mse(5, 110 + 0.001, x, y) - mse(5, 110, x, y)) / 0.001       # tiny step in b, w held still
print(f"✓ gradient for w: shortcut {kurz_w:.1f}, tiny step {probe_w:.1f}")   # nearly the same number
print(f"✓ gradient for b: shortcut {kurz_b:.1f}, tiny step {probe_b:.1f}")   # for b too

In [ ]:
def fit(x, y, lernrate=0.015, schritte=3000):     # the learning loop, with its default settings
    """Walk downhill: start with a flat line, then improve w and b step by step."""
    w, b = 0.0, float(np.mean(y))                     # start: flat line at the average demand
    verlauf = [(w, b, mse(w, b, x, y))]               # remember w, b and the MSE of every step
    for _ in range(schritte):                         # repeat this many times:
        gradient_w, gradient_b = gradienten(w, b, x, y)   # which way is downhill right now?

        w = w - lernrate * gradient_w                 # step w against its gradient
        b = b - lernrate * gradient_b                 # step b against its gradient

        verlauf.append((w, b, mse(w, b, x, y)))       # note where we are now
    return w, b, np.array(verlauf)                    # the final w and b, and the whole history


w_end, b_end, verlauf = fit(x, y)                     # run it: w_end, b_end are the final values
print(f"✓ Your loop found the line by itself: w = {w_end:.1f}, b = {b_end:.1f}, "   # the reward
      f"MSE = {verlauf[-1, 2]:.1f}. Your best by hand was 53.")

<details>
<summary>Answer</summary>

    w = w - lernrate * gradient_w      # step w against its gradient
    b = b - lernrate * gradient_b      # step b against its gradient

</details>

Your loop beat your hand for two reasons: it can use any `w`, not only whole numbers (that alone
brought you from 53 to 50.5 in step 7), and it moves `b` as well, which gets it to 50.3.

If there is time, watch it learn: press the **Play** button above the left plot. The left plot is a
map of all possible lines: across is `w`, up is `b`, and the colour is the MSE of that line, dark
is good. The point w = 8, b = 110 is your slider line from step 6, with MSE 53. The red path starts
at w = 0, b = 155.4, the flat line at the average, whose MSE is 227.4. You can skip this plot; the
cell after it, with the printed stops, is the one that matters.
The right plot shows the line settling into the cloud at the same time.

In [ ]:
from plotly.subplots import make_subplots             # two plots side by side

# the MSE for a grid of w and b values, for the coloured background on the left
W, B = np.meshgrid(np.linspace(0, 30, 60), np.linspace(60, 170, 60))   # 60 x 60 combinations of w and b
Z = np.array([[mse(w, b, x, y) for w in W[0]] for b in B[:, 0]])        # the MSE of each combination
# which steps to show: many at the start, where things change fast, fewer later
bilder = np.unique(np.geomspace(1, len(verlauf) - 1, 60).astype(int))   # Bilder: frames

fig = make_subplots(rows=1, cols=2, subplot_titles=("path through the MSE landscape",   # left and right
                                                    "the line in the data"))
fig.add_trace(go.Contour(x=W[0], y=B[:, 0], z=Z, colorscale="Viridis", showscale=False),   # coloured landscape
              row=1, col=1)
fig.add_trace(go.Scatter(x=[verlauf[0, 0]], y=[verlauf[0, 1]], mode="lines+markers",   # red path, starts at step 0
                         marker=dict(size=7, color="crimson"), name="w and b"), row=1, col=1)
fig.add_trace(go.Scatter(x=x, y=y, mode="markers", name="buildings",                   # buildings on the right
                         marker=dict(size=5, opacity=0.4, color="#4c78a8")), row=1, col=2)
fig.add_trace(go.Scatter(x=xs, y=verlauf[0, 0] * xs + verlauf[0, 1], mode="lines",     # the line at step 0
                         line=dict(width=4, color="crimson"), name="line"), row=1, col=2)
fig.frames = [go.Frame(name=str(i),                   # one picture per shown step
                       data=[go.Scatter(x=verlauf[:i + 1, 0], y=verlauf[:i + 1, 1],
                                        mode="lines+markers", marker=dict(size=5, color="crimson")),
                             go.Scatter(x=xs, y=verlauf[i, 0] * xs + verlauf[i, 1],
                                        mode="lines", line=dict(width=4, color="crimson"))],
                       layout=go.Layout(title=f"step {i}, MSE {verlauf[i, 2]:.1f}"),
                       traces=[1, 3])
              for i in bilder]
fig.update_layout(height=560, showlegend=False, title="step 0",   # size, and the Play and Stop buttons
                  updatemenus=[dict(type="buttons", x=0.02, y=1.15,
                                    buttons=[dict(label="Play", method="animate",
                                                  args=[None, dict(frame=dict(duration=80, redraw=True),
                                                                   fromcurrent=True)]),
                                             dict(label="Stop", method="animate",
                                                  args=[[None], dict(mode="immediate")])])])
fig.update_xaxes(title_text="w", row=1, col=1)                # horizontal axis, left: w
fig.update_yaxes(title_text="b", row=1, col=1)                # vertical axis, left: b
fig.update_xaxes(title_text="compactness", row=1, col=2)      # horizontal axis, right: compactness
fig.update_yaxes(title_text="heating demand", row=1, col=2)   # vertical axis, right: heating demand
fig.show()                                                    # draw it

The next cell prints a few stops along the way. Between which two printed steps does the MSE
first fall below 55?

In [ ]:
for schritt in (0, 10, 30, 100, 300, 1000, 3000):      # a few stops along the way (Schritt: step)
    w_s, b_s, f_s = verlauf[schritt]                   # w, b and MSE at that step
    print(f"step {schritt:5d}:  w = {w_s:5.2f},  b = {b_s:6.1f},  MSE = {f_s:6.1f}")   # one line per stop

<details>
<summary>Answer</summary>

Between step 300 (MSE 94.3) and step 1000 (MSE 52.4). Most of the 3000 steps are spent on the
last few points of improvement. Session 2 shows why it is so slow and how to make it fast.

</details>

## Step 9: what did the model learn?

Your MSE stops at about 50, and the test below checks that this is close to the best any straight
line can do. That is not the loop's fault: the heating demand was computed from the U-value **and** the compactness,
but the line only sees the compactness.

The map below colours each building by its **residual**: table minus predicted. If the table says
160 and the line says 150, the residual is +10: the line guessed too low (blue). A negative
residual means it guessed too high (red). Note that this is the miss from step 5 with the sign
turned round. If you chose a building in step 3, you get its result below the map.

In [ ]:
df["vorhersage"] = w_end * x + b_end                 # Vorhersage: what your line predicts
df["residuum"] = df["hwb"] - df["vorhersage"]        # Residuum: residual, table minus predicted

# map coloured by residual: blue = guessed too low, red = guessed too high
fig = px.scatter_map(df, lat="lat", lon="lon", color="residuum", zoom=11, height=560,   # dots coloured by residual
                     color_continuous_scale="RdBu", range_color=(-50, 50),   # colour scale from -50 to +50
                     hover_data=["id", "bauperiode", "u_wert", "kompaktheit", "vorhersage", "hwb"],
                     map_style="carto-positron")     # a light street map underneath
fig.show()                                            # draw it

if meine_id is not None and meine_id in set(df["id"]):           # only if you chose a building
    r = df.loc[df["id"] == meine_id, "residuum"].iloc[0]          # its residual
    richtung = "too low" if r > 0 else "too high"                 # richtung: direction
    print(f"Your building {meine_id}: residual {r:+.1f} kWh/m²a, so your line guessed {richtung}.")  # your result

Hover over a few dots and compare their `u_wert`. On average, buildings with U-value 1.50 (all of
them from before 1884) have a residual of +4.9: the line guesses too low for them. Buildings with
1.30 (1946 to 1976) have −7.3: too high. The 19 buildings after 1976, with 0.70, have −44.3. The
line misses in a pattern that follows the wall, and the wall comes from the construction period.

> Our model recovers a calculation rule that someone wrote down. It has not learned how a house
> behaves. Everything in this course stands or falls on that difference.

## Tests

Run this cell before you submit. It checks that your loop's MSE is less than 1.05 times the
smallest MSE any straight line can reach: if the best is 50, yours has to be below 52.5.

In [ ]:
w_t, b_t, verlauf_t = fit(x, y)                                      # run your loop again

eigener_fehler = float(np.mean((w_t * x + b_t - y) ** 2))            # eigener Fehler: your MSE
bestes_w, bestes_b = np.polyfit(x, y, 1)                             # bestes w, b: the best line, solved directly
bester_fehler = float(np.mean((bestes_w * x + bestes_b - y) ** 2))   # bester Fehler: its MSE

assert np.isfinite([w_t, b_t]).all(), "w or b ran away to infinity: check the minus signs"   # still numbers?
assert abs(eigener_fehler - verlauf_t[-1, 2]) < 1e-6, "the reported MSE does not match w and b"  # bookkeeping right?
assert verlauf_t[-1, 2] < verlauf_t[0, 2], "the MSE has to fall during the run"                  # did it improve?
assert eigener_fehler < 1.05 * bester_fehler, (                      # within 5 percent of the best
    f"still too far off: {eigener_fehler:.1f} against {bester_fehler:.1f} at best")
print(f"✓ All green. Your MSE {eigener_fehler:.2f}, the best possible {bester_fehler:.2f}.")   # reached only if all checks pass

## Where you stand now

You can say how wrong a line is in one number, find the direction that makes it better, and you
wrote the loop that walks there by itself. And you know what your result does not mean: it reproduces a calculation rule;
it has not learned how a house behaves.

Next session the model gets the U-value as a second input.

---

### Terms to learn

These words are in the flashcard app, together with those of the coming sessions, at
[gabriel-wurzer.github.io/data2/webapps/lernkarten](https://gabriel-wurzer.github.io/data2/webapps/lernkarten/).
Add it to your home screen; it works offline. *Lernkarten* is German for flashcards.

| term | in one sentence |
|---|---|
| model | a simplified rule that predicts an output from inputs; here, demand from compactness |
| neuron | inputs, one weight each, a bias, one output |
| weight, bias | the numbers the machine changes while it learns |
| parameter | the adjustable numbers of a model, such as `w` and `b` |
| loss | the general word for "how wrong the model is"; ours is the mean squared error |
| mean squared error (MSE) | the average of the squared misses |
| gradient | how much the loss changes per unit you move a parameter |
| gradient descent | repeatedly stepping against the gradient |
| learning rate | the number the gradient is multiplied by to give the step; too large and the loss rises |
| least squares | choosing the line whose squared misses add up to the least; Legendre and Gauss, around 1805 |
| residual | table (later: measured) minus predicted, for one building |

## Homework

Ten points, due before the next session: four for running it, six for your thoughts on it.

**How to do it**

1. Fill in the `YOUR TURN` gap above, yourself or from the solution notebook. If you took it from
   the solution, say so in one line at the top of your answers; that costs nothing.
2. Below, change only the lines under `# YOUR CHOICE`.
3. Write your answers into the last cell.
4. Choose Kernel, Restart Kernel and Run All Cells, check that everything runs, and upload the
   notebook in TUWEL.

**How it is graded**

- Four points for the run: it runs without an error (one), it shows what the task asks for (one),
  and every number you give about your run appears in your output or is worked out from it in a
  step you write down (two). Numbers the questions give you, examples from outside the notebook
  and forecasts you mark as your own expectation are exempt.
- Six points for your answers, one per question, no halves. Say what you saw and what it means,
  and you get the point, also with borrowed code. A sentence that would fit any model gets none.

### The task

1. **Steps that are too big.** In class the learning rate was 0.015. What if the steps are bigger?
   The first cell runs the loop for 300 steps at each learning rate in a list and tells you, for
   each, whether the MSE is still falling at the end or rising. Run it as it is: at 0.04 the MSE
   explodes, because each step jumps across the valley and lands higher on the other side. A loss
   curve that keeps rising usually means the learning rate is too large. Somewhere between 0.02
   and 0.04 it tips. Put in rates 0.001 apart, between the last one that falls and the first one
   that rises, and run again. Very large numbers are printed like `2.5e+135`: `2.5e+3` means
   2.5 × 1000 = 2500, and `2.5e+135` moves the decimal point 135 places to the right. You are done when you have two rates 0.001 apart, the lower one
   falling, the higher one rising. The second cell draws the curves on a logarithmic axis: each
   label is ten times the one below, so 10, 100 and 1000 sit at equal distances, and huge and
   small numbers fit in one picture. Keep that figure.
2. **How good is the line?** The third cell needs no change. It prints your loop's result next to
   the best possible line, and two numbers for question 1. The *typical miss* is the square root of
   the MSE, which brings it back to kWh/m²a. The *standard deviation* is the typical miss of the
   laziest model there is, one that always guesses the average. For three houses with demand 150,
   160 and 140, the average is 150, the misses are 0, 10 and −10, their squares 0, 100 and 100, the
   mean of those 66.7, and its square root 8.2: that is their standard deviation.
3. **Your building.** The fourth cell uses the building you chose in step 3 of the session. If you
   did not choose one, go back to step 3, paste an id and run that cell first.

In [ ]:
# YOUR CHOICE: the learning rates you try. Start with these, then close in, 0.001 apart.
lernraten = [0.01, 0.02, 0.03, 0.04]                  # lernraten: learning rates

laeufe = {}                                           # laeufe: the runs, one per learning rate
with np.errstate(over="ignore", invalid="ignore"):    # do not warn when numbers explode
    for lr in lernraten:                              # for every learning rate in the list:
        _, _, lauf = fit(x, y, lernrate=lr, schritte=300)     # lauf: one run's history
        laeufe[lr] = lauf                             # keep it for the figure
        ende, davor = lauf[-1, 2], lauf[-51, 2]       # ende: MSE at the end; davor: 50 steps before
        steigt = not np.isfinite(ende) or ende > davor        # steigt ("rises"): exploded, or going up?
        richtung = "rising" if steigt else "falling"  # richtung: direction
        print(f"learning rate {lr:<6}: MSE after 300 steps {ende:.4g}, "   # one line per rate
              f"over the last 50 steps {richtung}")

In [ ]:
# loss curves of all runs, logarithmic vertical axis
fig = go.Figure()                                     # an empty plot
for lr, lauf in laeufe.items():                       # one curve per learning rate
    fig.add_trace(go.Scatter(y=lauf[:, 2], mode="lines", name=f"learning rate {lr}"))   # its MSE curve
fig.update_layout(height=420, xaxis_title="step", yaxis_title="MSE",
                  yaxis=dict(type="log", dtick=1),       # one label for every factor of ten
                  title="where the run tips over")
fig.show()                                            # draw it

In [ ]:
bestes_w, bestes_b = np.polyfit(x, y, 1)              # bestes w, b: the best line, solved directly
fehler_loop = mse(w_end, b_end, x, y)                 # the MSE of your loop
print(f"your loop    : w = {w_end:.3f}, b = {b_end:.3f}, MSE {fehler_loop:.3f}")    # your result
print(f"best line    : w = {bestes_w:.3f}, b = {bestes_b:.3f}, "                  # the best possible
      f"MSE {mse(bestes_w, bestes_b, x, y):.3f}")
print(f"typical miss of your loop, the square root of its MSE: {np.sqrt(fehler_loop):.1f} kWh/m2a")  # np.sqrt: square root
print(f"standard deviation of the demand, the typical miss of always guessing the average: "
      f"{y.std():.1f} kWh/m2a")                                                  # y.std(): standard deviation
print(f"compactness in the data: {x.min():.2f} to {x.max():.2f}")                # smallest and largest
print(f"your line at compactness 15: {w_end * 15 + b_end:.1f} kWh/m2a, "         # outside the data
      f"highest demand in the data {y.max():.1f}")

In [ ]:
# meine_id is the building you chose in step 3 of the session
assert meine_id is not None, "choose your building in step 3 of the session first"   # stop if none chosen
haus = df[df["id"] == meine_id].iloc[0]               # haus: its row in the table
print(f"building {meine_id}, {haus['bauperiode']}, compactness {haus['kompaktheit']:.2f}, "   # its data
      f"u-value {haus['u_wert']:.2f}")
print(f"predicted {haus['vorhersage']:.1f}, computed {haus['hwb']:.1f}, "                      # its residual
      f"residual {haus['residuum']:+.1f} kWh/m2a (positive: the line guessed too low)")

nah = df[(df["kompaktheit"] - haus["kompaktheit"]).abs() < 0.25]   # nah: buildings of nearly the same compactness
print()                                                                       # an empty line
print(f"{len(nah)} buildings within 0.25 of its compactness:")                 # how many neighbours
print(f"  u-value from {nah['u_wert'].min():.2f} to {nah['u_wert'].max():.2f}, "   # their u-values
      f"average {nah['u_wert'].mean():.2f}")
print(f"  computed demand from {nah['hwb'].min():.1f} to {nah['hwb'].max():.1f}, "  # their demand
      f"average {nah['hwb'].mean():.1f}")

### Your interpretation

1. **Size.** Compare the typical miss of your line with the standard deviation; both are printed. How much better than always guessing the average is your line? Would you trust it to give the heating demand of one particular building to within 5 kWh/m²a?
2. **Sign and plausibility.** Your w is positive: a less compact building gets a higher demand. Say why that direction makes sense for a building. Then: is the size of w evidence from measured heating demand, or the result of fitting a line to computed values? Point to the sentence in this notebook that tells you.
3. **Units.** Give the units of w and of b. b is the line's value at compactness zero. Can a building have compactness zero? Say what that means for reading b on its own.
4. **Range.** Between which compactness values does your line rest on data? The third cell prints what it says at 15. Predicting outside the range of the data is called extrapolation. Do the buildings you fitted on support that number?
5. **Your building.** Give its residual with its sign, and say what the sign means. A higher u-value raises the computed demand, and the line knows nothing of it. Compare the u-value of your building with the average of its neighbours in compactness. Does that comparison explain the direction of the miss? If it does not, say so; that is a valid answer, as long as you say why it might not.
6. **Design consequence.** A colleague reads your w and concludes: lower a footprint's perimeter over square root of area by one, and you save that many kWh/m²a. Say what is right about that and what is wrong.

*Your text here.*